In [1]:
import re
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from math import log
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

pd.set_option('display.max_colwidth', 120)

In [2]:
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])
print(df.shape)
df.head()

(5572, 2)


,label,message
0,ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std t...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives around here though"


In [4]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess(text: str) -> list[str]:

    text = text.lower()

    text = re.sub(r'http\S+|www\.\S+', ' ', text)

    text = re.sub(r'[^a-z\s]', ' ', text)

    tokens = text.split()

    tokens = [lemmatizer.lemmatize(t) for t in tokens
              if t not in stop_words and len(t) > 2]
    return tokens

df['tokens'] = df['message'].apply(preprocess)
df[['message', 'tokens']].head()

,message,tokens
0,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...","[jurong, point, crazy, available, bugis, great, world, buffet, cine, got, amore, wat]"
1,Ok lar... Joking wif u oni...,"[lar, joking, wif, oni]"
2,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std t...,"[free, entry, wkly, comp, win, cup, final, tkts, may, text, receive, entry, question, std, txt, rate, apply]"
3,U dun say so early hor... U c already then say...,"[dun, say, early, hor, already, say]"
4,"Nah I don't think he goes to usf, he lives around here though","[nah, think, go, usf, life, around, though]"


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    df['tokens'], df['label'],
    test_size=0.2, random_state=42, stratify=df['label']
)
print(X_train.shape, X_test.shape, y_train.value_counts().to_dict())

(4457,) (1115,) {'ham': 3859, 'spam': 598}


In [8]:
class NaiveBayesSpam:
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.log_prior = {}
        self.log_likelihood = {}
        self.vocab = set()
        self.classes = []

    def fit(self, X, y):
        self.classes = list(set(y))
        for tokens in X:
            self.vocab.update(tokens)
        V = len(self.vocab)

        class_token_counts = {c: Counter() for c in self.classes}
        class_token_totals = {c: 0 for c in self.classes}
        class_doc_counts  = {c: 0 for c in self.classes}

        for tokens, label in zip(X, y):
            class_doc_counts[label] += 1
            for t in tokens:
                class_token_counts[label][t] += 1
                class_token_totals[label] += 1

        N = len(y)
        for c in self.classes:
            self.log_prior[c] = log((class_doc_counts[c] + self.alpha) / (N + self.alpha * len(self.classes)))  # likelihood
            ll = {}
            denom = class_token_totals[c] + self.alpha * V
            for w in self.vocab:
                ll[w] = log((class_token_counts[c][w] + self.alpha) / denom)
            self.log_likelihood[c] = ll
        return self

    def _score(self, tokens, c):
        s = self.log_prior[c]
        for w in tokens:
            if w in self.vocab:
                s += self.log_likelihood[c][w]
        return s

    def predict(self, X):
        preds = []
        for tokens in X:
            scores = {c: self._score(tokens, c) for c in self.classes}
            preds.append(max(scores, key=scores.get))
        return np.array(preds)

    def predict_proba(self, X):
        probs = []
        for tokens in X:
            scores = np.array([self._score(tokens, c) for c in self.classes])
            scores -= scores.max()
            p = np.exp(scores) / np.exp(scores).sum()
            probs.append(p)
        return np.array(probs)

In [9]:
model = NaiveBayesSpam(alpha=1.0)
model.fit(X_train.tolist(), y_train.tolist())

y_pred = model.predict(X_test.tolist())

print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9820627802690582
[[958   8]
 [ 12 137]]
              precision    recall  f1-score   support

         ham       0.99      0.99      0.99       966
        spam       0.94      0.92      0.93       149

    accuracy                           0.98      1115
   macro avg       0.97      0.96      0.96      1115
weighted avg       0.98      0.98      0.98      1115



In [10]:
def top_words(model, label, n=15):
    lls = model.log_likelihood[label]
    # сортируем по лог-вероятности
    return sorted(lls.items(), key=lambda x: x[1], reverse=True)[:n]

print("Топ спам-слов:")
for w, lp in top_words(model, 'spam'):
    print(f"  {w:15s}  logP={lp:.2f}")

print("\nТоп ham-слов:")
for w, lp in top_words(model, 'ham'):
    print(f"  {w:15s}  logP={lp:.2f}")

Топ спам-слов:
  call             logP=-3.80
  free             logP=-4.31
  txt              logP=-4.61
  text             logP=-4.81
  mobile           logP=-4.89
  stop             logP=-4.91
  claim            logP=-4.95
  reply            logP=-5.05
  min              logP=-5.27
  prize            logP=-5.28
  get              logP=-5.32
  cash             logP=-5.34
  tone             logP=-5.34
  service          logP=-5.37
  send             logP=-5.48

Топ ham-слов:
  get              logP=-4.87
  call             logP=-5.02
  got              logP=-5.09
  day              logP=-5.09
  know             logP=-5.11
  come             logP=-5.11
  good             logP=-5.16
  time             logP=-5.25
  love             logP=-5.25
  like             logP=-5.26
  want             logP=-5.34
  need             logP=-5.47
  home             logP=-5.49
  sorry            logP=-5.51
  lor              logP=-5.51


In [12]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ('vec', CountVectorizer(tokenizer=lambda x: x, preprocessor=lambda x: x,
                            token_pattern=None)),
    ('nb', MultinomialNB(alpha=1.0))
])

pipe.fit(X_train.tolist(), y_train.tolist())
y_pred_sk = pipe.predict(X_test.tolist())

print("sklearn Accuracy:", accuracy_score(y_test, y_pred_sk))
print(classification_report(y_test, y_pred_sk))

sklearn Accuracy: 0.9820627802690582
              precision    recall  f1-score   support

         ham       0.99      0.99      0.99       966
        spam       0.94      0.92      0.93       149

    accuracy                           0.98      1115
   macro avg       0.97      0.96      0.96      1115
weighted avg       0.98      0.98      0.98      1115

